# FMCG Supply Chain Analytics

## Warehouse-Level Product Weight Prediction

**Objective:** analyze warehouse-level operational characteristics and evaluate whether they can predict `product_wg_ton`, the product weight handled by each warehouse.

This notebook is intentionally structured as a business-analysis workflow: **business problem → data quality → EDA → leakage check → modeling → interpretation**.

## 1. Business Context

The dataset represents warehouse-level information from an instant noodles FMCG supply network. Variables cover warehouse capacity, location, distribution reach, operational issues, infrastructure, compliance and supply activity.

The central analytical question is:

> Can warehouse-level characteristics be used to predict product weight handled by a warehouse?

A key modeling concern is whether any feature contains information that would only be known after the target outcome or is mechanically related to it. This is checked explicitly before interpreting model performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('../data/FMCG_data.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Dataset Overview

In [ ]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.info()

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

### Missing values

The original dataset contains missing values in `workers_num`, `wh_est_year`, and `approved_wh_govt_certificate`. For the modeling pipeline, numerical variables are median-imputed and categorical variables are most-frequent-imputed. `wh_est_year` is excluded because of its substantial missingness in the supplied dataset.

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['product_wg_ton'], bins=40)
plt.title('Distribution of Product Weight')
plt.xlabel('Product Weight (ton)')
plt.ylabel('Warehouses')
plt.show()

In [ ]:
zone_counts = df['zone'].value_counts()
plt.figure(figsize=(8,5))
plt.bar(zone_counts.index, zone_counts.values)
plt.title('Warehouse Distribution by Zone')
plt.xlabel('Zone')
plt.ylabel('Warehouses')
plt.show()

In [ ]:
numeric_corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(11,8))
plt.imshow(numeric_corr, cmap='viridis', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(numeric_corr)), numeric_corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(numeric_corr)), numeric_corr.columns, fontsize=7)
plt.title('Numerical Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Leakage / Feature Validity Check

The original analysis identified a very strong correlation between `storage_issue_reported_l3m` and `product_wg_ton`. The correlation is approximately **0.987** in the supplied dataset.

This is unusually high for an operational predictor, so it should not automatically be interpreted as a causal relationship. We investigate the feature before relying on it for model performance.

In [ ]:
corr_storage = df['storage_issue_reported_l3m'].corr(df['product_wg_ton'])
print(f'Correlation: {corr_storage:.6f}')
print(df.groupby('storage_issue_reported_l3m')['product_wg_ton'].agg(['count','mean','median','min','max']).head(10))

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=42)
plt.figure(figsize=(9,5))
plt.scatter(sample['storage_issue_reported_l3m'], sample['product_wg_ton'], s=8, alpha=0.25)
plt.title('Storage Issues vs Product Weight')
plt.xlabel('Storage Issues Reported (last 3 months)')
plt.ylabel('Product Weight (ton)')
plt.show()

### Interpretation

The feature is strongly associated with the target in this dataset. However, such a strong relationship can produce an overly optimistic model if the feature is unavailable at prediction time or is derived from the same underlying process as the target. Therefore, two modeling views are reported:

1. **Storage-inclusive model:** shows predictive performance when all supplied operational features are available.
2. **Leakage-aware model:** removes `storage_issue_reported_l3m` to test how much predictive power remains without the suspicious feature.

This distinction is important when presenting the model as a real supply-planning solution.

## 5. Modeling Setup

In [ ]:
TARGET = 'product_wg_ton'
DROP_COLS = ['product_wg_ton', 'Ware_house_ID', 'WH_Manager_ID', 'wh_est_year']
X = df.drop(columns=DROP_COLS)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

def evaluate(model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mean_squared_error(y_test, pred),
        'R2': r2_score(y_test, pred)
    }, pred

## 6. Linear Regression Baseline

In [ ]:
linear = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_metrics, linear_pred = evaluate(linear, X_train, X_test)
linear_metrics

## 7. Random Forest Regression

In [ ]:
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_metrics, rf_pred = evaluate(rf, X_train, X_test)
rf_metrics

## 8. Leakage-Aware Random Forest

Remove `storage_issue_reported_l3m` and repeat the same modeling procedure. This is not necessarily the final production feature set; it is a diagnostic to quantify dependence on the suspicious variable.

In [ ]:
X_no_storage = X.drop(columns=['storage_issue_reported_l3m'])
Xns_train, Xns_test, yns_train, yns_test = train_test_split(
    X_no_storage, y, test_size=0.20, random_state=42
)

cat_ns = X_no_storage.select_dtypes(include='object').columns.tolist()
num_ns = X_no_storage.select_dtypes(exclude='object').columns.tolist()
pre_ns = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_ns),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_ns)
])

rf_ns = Pipeline([
    ('preprocessor', pre_ns),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])
rf_ns.fit(Xns_train, yns_train)
rf_ns_pred = rf_ns.predict(Xns_test)
rf_ns_metrics = {
    'MAE': mean_absolute_error(yns_test, rf_ns_pred),
    'MSE': mean_squared_error(yns_test, rf_ns_pred),
    'R2': r2_score(yns_test, rf_ns_pred)
}
rf_ns_metrics

## 9. Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Linear Regression', **linear_metrics},
    {'Model': 'Random Forest — storage included', **rf_metrics},
    {'Model': 'Random Forest — storage excluded', **rf_ns_metrics}
])
comparison

### Observed results

On the supplied data and the fixed 80/20 split (`random_state=42`), the storage-inclusive Random Forest is highly predictive. Removing `storage_issue_reported_l3m` causes a large drop in predictive performance. This indicates that the feature carries a very large amount of information about the target.

The correct business interpretation is **not** that storage issues cause product weight. The dataset should be investigated further to determine how this variable was generated and whether it is legitimately available before the prediction decision.

## 10. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, rf_pred, s=8, alpha=0.25)
lims = [min(y_test.min(), rf_pred.min()), max(y_test.max(), rf_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('Actual Product Weight')
plt.ylabel('Predicted Product Weight')
plt.title('Random Forest — Storage Feature Included')
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(yns_test, rf_ns_pred, s=8, alpha=0.25)
lims = [min(yns_test.min(), rf_ns_pred.min()), max(yns_test.max(), rf_ns_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('Actual Product Weight')
plt.ylabel('Predicted Product Weight')
plt.title('Random Forest — Leakage-Aware Model')
plt.show()

## 11. Conclusion

The analysis demonstrates a strong relationship between warehouse-level operational variables and product weight in the supplied FMCG dataset. The original model achieved very high predictive performance largely because `storage_issue_reported_l3m` is exceptionally correlated with the target.

For a production-style supply planning application, the next step should be to establish the timing and business definition of this variable, then build a feature set containing only information known **before** the prediction point.

### Potential extensions

- Time-aware demand forecasting
- Warehouse capacity constraints
- Safety-stock and reorder-point calculations
- Transportation-cost modeling
- Warehouse network optimization
- Scenario and sensitivity analysis
- SHAP-based model explainability
- Power BI decision dashboard